In [1]:
!pip install simpletransformers
!pip install transformers-interpret

import ast
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from simpletransformers.classification import MultiLabelClassificationModel
from transformers_interpret import SequenceClassificationExplainer

In [2]:
trainPath = "../input/multilabel-data/trainDataGoEML.csv"
testPath = "../input/multilabel-data/testDataGoEML.csv"

emoList = []
train_df = pd.read_csv(trainPath)
for label in train_df['label']:
    emoList.extend(ast.literal_eval(label))
test_df = pd.read_csv(testPath)
for label in test_df['label']:
    emoList.extend(ast.literal_eval(label))
emoList = list(set(emoList))
    
train_df = pd.read_csv(trainPath)
trainCol = []
for label in train_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    trainCol.append(tempL)
train_df['label'] = trainCol
train_df = train_df[train_df.label != 'neutral']
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)
    
test_df = pd.read_csv(testPath)
testCol = []
for label in test_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    testCol.append(tempL)
test_df['label'] = testCol
test_df = test_df[test_df.label != 'neutral']
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

df = pd.concat([train_df, test_df], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)

In [3]:
model = MultiLabelClassificationModel('roberta', 'roberta-base', num_labels=len(emoList), args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,
                                                                                     'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                                                                     'num_train_epochs': 1, "reprocess_input_data": True, 'max_seq_length': 256})

model.train_model(df)
# result, model_outputs, wrong_predictions = model.eval_model(test_df)

# textList = []
# for text in test_df['text']:
#     textList.append(text)
# predictions, raw_outputs = model.predict(textList)
# metrics.f1_score(testCol, predictions, average='macro')

In [4]:
train_df = pd.read_csv("../input/multilabel-data/trainSEDataML.csv")
textList = []
for text in train_df['sentence']:
    textList.append(text)
predictions, raw_outputs = model.predict(textList)

emoCol = []
for pred in predictions:
    tempList = []
    for index in np.argwhere(np.array(pred)):
        tempList.append(emoList[index[0]])
    emoCol.append(tempList)
    
train_df['emotions'] = emoCol
train_df.to_csv('trainMo.csv', index=False)

In [5]:
test_df = pd.read_csv("../input/multilabel-data/testSEDataML.csv")
textList = []
for text in test_df['sentence']:
    textList.append(text)
predictions, raw_outputs = model.predict(textList)

emoCol = []
for pred in predictions:
    tempList = []
    for index in np.argwhere(np.array(pred)):
        tempList.append(emoList[index[0]])
    emoCol.append(tempList)
    
test_df['emotions'] = emoCol
test_df.to_csv('testMo.csv', index=False)

In [6]:
trainPath = "../input/multilabel-data/trainSEDataML.csv"
testPath = "../input/multilabel-data/testSEDataML.csv"

emoList = []
train_df = pd.read_csv(trainPath)
for label in train_df['label']:
    emoList.extend(ast.literal_eval(label))
test_df = pd.read_csv(testPath)
for label in test_df['label']:
    emoList.extend(ast.literal_eval(label))
emoList = list(set(emoList))
    
train_df = pd.read_csv(trainPath)
trainCol = []
for label in train_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    trainCol.append(tempL)
train_df['label'] = trainCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)
    
test_df = pd.read_csv(testPath)
testCol = []
for label in test_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    testCol.append(tempL)
test_df['label'] = testCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

In [7]:
pre_model = MultiLabelClassificationModel('roberta', 'roberta-base', num_labels=len(emoList), args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,
                                                                                     'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                                                                     'num_train_epochs': 2, "reprocess_input_data": True, 'max_seq_length': 256})
pre_model.train_model(train_df)
result, model_outputs, wrong_predictions = pre_model.eval_model(test_df)
print(result)

textList = []
for text in test_df['text']:
    textList.append(text)
predictions, raw_outputs = pre_model.predict(textList)
f1_score(testCol, predictions, average='macro')

In [9]:
trainPath = "./trainMo.csv"
testPath = "./trainMo.csv"

emoList = []
train_df = pd.read_csv(trainPath)
for label in train_df['label']:
    emoList.extend(ast.literal_eval(label))
test_df = pd.read_csv(testPath)
for label in test_df['label']:
    emoList.extend(ast.literal_eval(label))
emoList = list(set(emoList))
    
train_df = pd.read_csv(trainPath)
trainCol = []
for label in train_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    trainCol.append(tempL)
train_df['label'] = trainCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)
train_df['text'] = train_df['text'] + ' ' + train_df['emotions']
    
test_df = pd.read_csv(testPath)
testCol = []
for label in test_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in ast.literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    testCol.append(tempL)
test_df['label'] = testCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)
test_df['text'] = test_df['text'] + ' ' + test_df['emotions']

In [10]:
post_model = MultiLabelClassificationModel('roberta', 'roberta-base', num_labels=len(emoList), args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,
                                                                                     'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                                                                     'num_train_epochs': 2, "reprocess_input_data": True, 'max_seq_length': 256})
post_model.train_model(train_df)
result, model_outputs, wrong_predictions = post_model.eval_model(test_df)
print(result)

textList = []
for text in test_df['text']:
    textList.append(text)
predictions, raw_outputs = post_model.predict(textList)
f1_score(testCol, predictions, average='macro')

In [12]:
test_df = pd.read_csv("../input/multilabel-data/testSEDataML.csv")
pre_textList = []
for text in test_df['sentence']:
    pre_textList.append(text)
    
test_df = pd.read_csv("./testMo.csv")
test_df['sentence'] = test_df['sentence'] + ' ' + test_df['emotions']
post_textList = []
for text in test_df['sentence']:
    post_textList.append(text)
    
trueLab = []
for label in test_df['label']:
    trueLab.append(ast.literal_eval(label))
    
pre_cls_explainer = SequenceClassificationExplainer(pre_model.model, pre_model.tokenizer, 
                                                custom_labels=emoList)
post_cls_explainer = SequenceClassificationExplainer(post_model.model, post_model.tokenizer, 
                                                custom_labels=emoList)

In [36]:
idx = 76

print(f"Original Text : {pre_textList[idx]}")
print('-'*100)
print(f"Text with Emotion : {post_textList[idx]}")
print('-'*100)
print(f"True Label : {trueLab[idx]}")
print('-'*100)

word_attributions = pre_cls_explainer(pre_textList[idx])
pred_idx = pre_cls_explainer.predicted_class_index
print(f"Prediction without Emotions : {emoList[int(pred_idx)]}")
# print(word_attributions)
print('-'*100)
word_attributions = post_cls_explainer(post_textList[idx])
pred_idx = post_cls_explainer.predicted_class_index
print(f"Prediction with Emotions : {emoList[int(pred_idx)]}")
# print(word_attributions)